In [ ]:
import warnings

import pandas as pd
import statsmodels.api as sm
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import MinMaxScaler


### Get File path and load the csv data

In [ ]:
from util.data_path import cassava_path

In [ ]:
wide = pd.read_csv(cassava_path)
long = wide.melt(id_vars="year", var_name="month", value_name="price").sort_values(
    ["year", "month"]
)

long["date"] = pd.to_datetime(
    (long["year"] - 543).astype(str)
    + "-"
    + long["month"].astype(str).str.zfill(2)
    + "-01"
)
long = long.set_index("date").sort_index()

### Set the training dataset and the predicted data

In [ ]:
gregorian_2566 = 2023
gregorian_2567 = 2024

train = long[long.index.year <= gregorian_2566]["price"].to_frame()
future = long[long.index.year == gregorian_2567]["price"].to_frame()

scaler = MinMaxScaler()
train["scaled"] = scaler.fit_transform(train[["price"]])

series = train["scaled"].values.astype(np.float32)

In [ ]:
y_train = train["price"].asfreq("MS")
y_test = future["price"].asfreq("MS")

In [ ]:
SEQ_LEN = 12
PRED_LEN = 12

### Training the ARIMA model using uto_arima

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    stepwise_fit = auto_arima(
        y_train,
        start_p=0,
        start_q=0,
        max_p=3,
        max_q=3,
        seasonal=True,
        m=12,  # monthly seasonality
        trace=True,  # view the search trace
        error_action="ignore",
        suppress_warnings=True,
        information_criterion="aicc",
    )

In [ ]:
print(f"Selected order   : {stepwise_fit.order}")
print(f"Selected seasonal: {stepwise_fit.seasonal_order}")

### Fit final model & forecast 12 steps

In [ ]:
model_arima = sm.tsa.ARIMA(
    y_train,
    order=stepwise_fit.order,
    seasonal_order=stepwise_fit.seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit()

arima_forecast = model_arima.forecast(steps=PRED_LEN)
arima_forecast.name = "ARIMA_forecast"

# 4. Quick evaluation
mae_arima = mean_absolute_error(y_test, arima_forecast)
print(f"\nARIMA MAE 2567 = {mae_arima:.4f}")

# (Optional) display side-by-side
display(pd.concat([y_test.rename("actual"), arima_forecast], axis=1))